In [1]:
library(dplyr)
library(tibble) # <--- Esta es la que te falta
library(limma)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




## Expresión diferencial (Limma)

In [11]:
# 2. Leer la matriz
cat("Leyendo matriz_expresion_normalizada.csv...\n")
df <- read.csv("matriz_expresion_normalizada.csv", check.names = FALSE)

# --- PASO DE LIMPIEZA ANTIESTRÉS ---
# 1. Eliminamos columnas que no tengan nombre (columnas vacías al final del CSV)
df <- df[, colnames(df) != "" & !is.na(colnames(df))]

Leyendo matriz_expresion_normalizada.csv...


In [12]:
matriz <- df %>%
  filter(!is.na(SYMBOL) & SYMBOL != "") %>%
  tibble::column_to_rownames("SYMBOL") %>%  # <--- Agregá el prefijo tibble::
  as.matrix()

In [13]:
# 3. Identificar grupos (Male vs Female) por el nombre de las columnas
col_names <- colnames(matriz)
grupos <- ifelse(grepl("Female", col_names, ignore.case = TRUE), "Female", "Male")
grupos <- factor(grupos, levels = c("Male", "Female")) # Male es el control (referencia)

cat("Muestras detectadas:\n")
print(table(grupos))

Muestras detectadas:
grupos
  Male Female 
    21     16 


In [14]:
# 4. Configurar el diseño experimental para limma
design <- model.matrix(~0 + grupos)
colnames(design) <- levels(grupos)

In [15]:
# 5. Ajuste del modelo lineal y Contraste
# Comparamos Female contra Male (Female - Male)
fit <- lmFit(matriz, design)
cont_matrix <- makeContrasts(FemaleVsMale = Female - Male, levels = design)
fit2 <- contrasts.fit(fit, cont_matrix)
fit2 <- eBayes(fit2)

In [16]:
# 6. Extraer resultados y generar el archivo .rnk
res <- topTable(fit2, coef = "FemaleVsMale", number = Inf, adjust.method = "BH")

# El archivo .rnk para GSEA usa el Gen y una métrica de ranking (usamos 't')
ranking_gsea <- res %>%
  tibble::rownames_to_column("SYMBOL") %>%
  select(SYMBOL, t) %>%
  arrange(desc(t))

# 7. Guardar el archivo final
write.table(ranking_gsea, "placenta_sex_ranking.rnk", 
            sep = "\t", row.names = FALSE, quote = FALSE, col.names = FALSE)

cat("✅ ¡Listo! Se ha generado 'placenta_sex_ranking.rnk'.\n")

✅ ¡Listo! Se ha generado 'placenta_sex_ranking.rnk'.


## Filtrar expresion por cuantiles para aumentar significancia

In [2]:
library(dplyr)
library(tibble)
library(limma)

# 1. Leer los datos
df <- read.csv("matriz_expresion_normalizada.csv", check.names = FALSE)

# 2. Aplicar el filtro de cuantil (Q1)
matriz_num <- df %>%
  filter(!is.na(SYMBOL) & SYMBOL != "") %>%
  column_to_rownames("SYMBOL") %>%
  as.matrix()

# Calculamos el corte del 25% (Q1)
umbral_q1 <- quantile(matriz_num, 0.25)

# Nos quedamos con genes que superan el Q1 en al menos el 50% de las muestras
genes_a_mantener <- rowSums(matriz_num > umbral_q1) >= (ncol(matriz_num) / 2)
matriz_filtrada <- matriz_num[genes_a_mantener, ]

cat("Genes antes:", nrow(matriz_num), " -> Genes después:", nrow(matriz_filtrada), "\n")

# 3. Análisis diferencial rápido para sacar el logFC
grupos <- factor(ifelse(grepl("Female", colnames(matriz_filtrada), ignore.case = TRUE), "Female", "Male"), 
                 levels = c("Male", "Female"))

design <- model.matrix(~0 + grupos)
colnames(design) <- levels(grupos)
fit <- lmFit(matriz_filtrada, design)
cont_matrix <- makeContrasts(FemaleVsMale = Female - Male, levels = design)
fit2 <- contrasts.fit(fit, cont_matrix)
fit2 <- eBayes(fit2)

# 4. Crear el archivo .rnk basado en logFC
res <- topTable(fit2, coef = "FemaleVsMale", number = Inf)
ranking_gsea <- res %>%
  rownames_to_column("SYMBOL") %>%
  select(SYMBOL, logFC) %>%
  arrange(desc(logFC))

write.table(ranking_gsea, "placenta_limpia_hallmark.rnk", 
            sep="\t", row.names=F, col.names=F, quote=F)

Genes antes: 19699  -> Genes después: 19696 


## Ranking agresivo

In [3]:
# En R, partiendo de tu tabla de resultados 'res' que ya tenías:
ranking_agresivo <- res %>%
  rownames_to_column("SYMBOL") %>%
  # Creamos el score combinando magnitud (logFC) y significancia (P.Value)
  mutate(score = sign(logFC) * -log10(P.Value)) %>% 
  select(SYMBOL, score) %>%
  arrange(desc(score))

write.table(ranking_agresivo, "placenta_agresiva.rnk", 
            sep="\t", row.names=F, col.names=F, quote=F)

## Analisis cualitativo

In [4]:
# 1. Lista expandida de genes (Director + Eferocitosis + Metabo)
genes_finales <- c(
  # Pedidos por el director
  "ITGAX", "CD86", "IL10", "FCGR3A", "FCGR3B", 
  # Eferocitosis (Limpieza celular)
  "MERTK", "AXL", "TYRO3", "TIMD4", "C1QTNF5",
  # Inflamación (Pro y Anti)
  "TNF", "IL1B", "IL6", "TGFB1", "CCL2",
  # Metabolismo Placentario
  "SLC2A1", "FABP4", "LPL", "PPARG", "CPT1A"
)

# 2. Filtrado y creación de la tabla
tabla_tesis <- res %>%
  tibble::rownames_to_column("SYMBOL") %>%
  filter(SYMBOL %in% genes_finales) %>%
  mutate(
    # Calculamos el Score Agresivo
    Score = sign(logFC) * -log10(P.Value),
    # Traducimos el signo a idioma humano
    Expresion = ifelse(logFC > 0, "Más en Hembras (F)", "Más en Varones (M)")
  ) %>%
  select(SYMBOL, Expresion, Score, logFC, P.Value) %>%
  arrange(Score) # Ordenados de Varones a Hembras

# 3. Mostrar en consola
print("--- TABLA DE GENES CLAVE PARA LA TESIS ---")
print(tabla_tesis)

# 4. Guardar a CSV para pasarle a tu director
write.csv(tabla_tesis, "tabla_genes_clave_placenta.csv", row.names = FALSE)

[1] "--- TABLA DE GENES CLAVE PARA LA TESIS ---"
    SYMBOL          Expresion       Score       logFC   P.Value
1     CCL2 Más en Varones (M) -0.35747941 -1.11872856 0.4390567
2  C1QTNF5 Más en Varones (M) -0.35231714 -1.10593892 0.4443067
3    FABP4 Más en Varones (M) -0.30858742 -0.99487872 0.4913745
4    MERTK Más en Varones (M) -0.30477972 -0.98496674 0.4957016
5     IL10 Más en Varones (M) -0.26206884 -0.87089344 0.5469293
6   FCGR3A Más en Varones (M) -0.25616025 -0.85467222 0.5544211
7    TYRO3 Más en Varones (M) -0.24828227 -0.83286795 0.5645699
8     CD86 Más en Varones (M) -0.23624070 -0.79913974 0.5804426
9    PPARG Más en Varones (M) -0.20525330 -0.70999257 0.6233712
10     IL6 Más en Varones (M) -0.18927410 -0.66260769 0.6467343
11  SLC2A1 Más en Varones (M) -0.17712950 -0.62590461 0.6650748
12   ITGAX Más en Varones (M) -0.13981844 -0.50910727 0.7247389
13     TNF Más en Varones (M) -0.12946160 -0.47552082 0.7422298
14  FCGR3B Más en Varones (M) -0.12028317 -0.44529970 0